In [4]:

# Baixar a malha municipal
municipios = read_municipality(
    year=2024,
    simplified=False
)

# Diagnóstico da versão e do objeto
print("Tipo:", type(municipios))
print("CRS atual:", municipios.crs)
print("Colunas:", municipios.columns.tolist())

# Compatibilidade com versões antigas do GeoPandas
if municipios.crs is None:
    municipios.crs = "EPSG:4674"

# Projetar para sistema métrico e calcular centroides
municipios_proj = municipios.to_crs(epsg=5880)
centroides = municipios_proj.geometry.centroid

# Transformar de volta para latitude/longitude
centroides_geo = gpd.GeoSeries(
    centroides,
    crs="EPSG:5880"
).to_crs(epsg=4674)

municipios["latitude"] = centroides_geo.y
municipios["longitude"] = centroides_geo.x

print(
    municipios[
        ["name_muni", "abbrev_state", "latitude", "longitude"]
    ].head()
)


Tipo: <class 'geopandas.geodataframe.GeoDataFrame'>
CRS atual: None
Colunas: ['code_muni', 'name_muni', 'code_state', 'abbrev_state', 'name_state', 'code_region', 'name_region', 'geometry']


/Users/marcosgregorio/codigos/pgp_trabalho/.venv/lib/python3.8/site-packages/geopandas/array.py:93: ShapelyDeprecationWarning: __len__ for multi-part geometries is deprecated and will be removed in Shapely 2.0. Check the length of the `geoms` property instead to get the  number of parts of a multi-part geometry.
  aout[:] = out
/Users/marcosgregorio/codigos/pgp_trabalho/.venv/lib/python3.8/site-packages/geopandas/array.py:546: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  return _unary_geo("centroid", self)
/Users/marcosgregorio/codigos/pgp_trabalho/.venv/lib/python3.8/site-packages/geopandas/geoseries.py:128: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  data = from_shapely(s.values)
/Users/marcosgregorio/codigos/pgp_trabalho/.venv/lib/python3.8/site-packages/geopandas/geoseries

               name_muni abbrev_state   latitude  longitude
0  Alta Floresta D'oeste           RO -12.469787 -62.274391
1              Ariquemes           RO  -9.951937 -62.957505
2                 Cabixi           RO -13.474909 -60.639773
3                 Cacoal           RO -11.301322 -61.324675
4             Cerejeiras           RO -13.203692 -61.261150


In [5]:

# Preparar tabela de coordenadas
coordenadas = municipios[
    ["name_muni", "abbrev_state", "latitude", "longitude"]
].copy()

coordenadas = coordenadas.rename(columns={
    "name_muni": "MUNICIPIO",
    "abbrev_state": "UF"
})

# Normalizar nomes para garantir correspondência
for coluna in ["MUNICIPIO", "UF"]:
    coordenadas[coluna] = (
        coordenadas[coluna]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.upper()
    )

# Remover eventuais duplicatas
coordenadas = coordenadas.drop_duplicates(
    subset=["UF", "MUNICIPIO"]
)

# Associar latitude e longitude ao dataset original
df = df.merge(
    coordenadas,
    on=["UF", "MUNICIPIO"],
    how="left",
    validate="many_to_one"
)

# Verificar municípios sem correspondência
sem_coordenadas = (
    df.loc[df["latitude"].isna(), ["UF", "MUNICIPIO"]]
    .drop_duplicates()
)

print("Total de registros:", len(df))
print("Registros sem coordenadas:", df["latitude"].isna().sum())
print("\nMunicípios sem correspondência:")
print(sem_coordenadas.to_string(index=False))

# Salvar o resultado
df.to_csv(
    "../results/dataset_com_clusters_hdbscan_coordenadas.csv",
    index=False,
    encoding="utf-8"
)

print("\nArquivo salvo com sucesso!")


Total de registros: 85851
Registros sem coordenadas: 786

Municípios sem correspondência:
UF                 MUNICIPIO
MG               PASSA-VINTE
RN                 BOA SAÚDE
RS                 MAÇAMBARA
SC                 GRÃO PARÁ
SC                  PIÇARRAS
                            
PA      SANTA ISABEL DO PARÁ
RS     SANTANA DO LIVRAMENTO
ES           ATILIO VIVACQUA
PI         BARRA D´ALCÂNTARA
SP     SANTA BÁRBARA D´OESTE
RO     ALTA FLORESTA D´OESTE
SP           GUARANI D´OESTE
BA           SANTA TERESINHA
RO           ESPIGÃO D´OESTE
MT SANTO ANTÔNIO DO LEVERGER
RS             RESTINGA SECA
AL    OLHO D´ÁGUA DAS FLORES
CE                   ACARAPÉ
RO          ALVORADA D´OESTE
RO        MACHADINHO D´OESTE
PR          DIAMANTE D´OESTE
AL     OLHO D´ÁGUA DO CASADO
AL        OLHO D´ÁGUA GRANDE
AP   PEDRA BRANCA DO AMAPARÍ
BA              DIAS D´ÁVILA
BA                      IUIÚ
BA                 JEQUIRIÇÁ
GO        SÃO JOÃO D´ALIANÇA
MA             PINDARÉ MIRIM
MG       BA